# Regularized Hartley Neural Operator (RHNO)
### HNO vs RHNO Comparison Experiments

**Paper:** *When Real Beats Complex: Hartley Neural Operators and the Role of Spectral Basis Alignment in PDE Learning*  
**Repo:** https://github.com/jaysulk/RFHT

This notebook:
1. Clones the repo and mounts Google Drive for result persistence
2. Runs the parameter count analysis (no GPU needed)
3. Trains HNO (dense, paper baseline) vs RHNO (structured-correction) on Poisson and Heat equations
4. Saves all results to Drive as `.pt` (model checkpoints) and `.pkl` (histories/metrics)
5. Plots training curves and error comparison

> **Runtime:** Set to GPU (T4 recommended). Runtime → Change runtime type → T4 GPU


## 0. Environment Check

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print("GPU:", result.stdout.strip())
else:
    print("⚠️  No GPU detected — training will be slow. Switch runtime to GPU.")

# Check PyTorch
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/RHNO_experiments'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved to: {SAVE_DIR}")


## 2. Clone Repository

In [ ]:
import os

REPO_DIR = '/content/RFHT'

if os.path.exists(REPO_DIR):
    print("Repo already cloned, pulling latest...")
    os.chdir(REPO_DIR)
    os.system('git pull')
else:
    os.system('git clone https://github.com/jaysulk/RFHT.git')

os.chdir(REPO_DIR)
print("\nRepo contents:")
for f in sorted(os.listdir('.')):
    print(f"  {f}")

# Add repo root to path (flat structure — all modules at root)
sys.path.insert(0, REPO_DIR)


In [ ]:
# Overwrite rfht.py and rhno.py with the CORRECTED v2 architecture:
#  - diagonal-capable dense weights (preserves elliptic advantage)
#  - zero-init structured butterfly correction (RHNO == HNO at init)
#  - training.py gradient_error fix applied separately below
RFHT_SRC = '"""\nRegularized Fast Hartley Transform (RFHT) for Neural Operators — CORRECTED\n==========================================================================\nInspired by Jones (2022), "The Regularized Fast Hartley Transform."\n\nCORRECTED DESIGN (v2):\n----------------------\nThe previous version made a conceptual error: it replaced the *learned*\nspectral weights with a butterfly factorization. This crippled the diagonal\noperator that elliptic PDEs require (pointwise mult by 1/|k|^2), hurting\nexactly the Poisson/biharmonic case where HNO should dominate.\n\nThis version separates the two distinct ideas in Jones\' book:\n\n  (A) RFHT as a FAST TRANSFORM\n      The butterfly with FIXED twiddle factors computes the DHT efficiently.\n      In PyTorch this is mathematically identical to the Re-Im FFT trick\n      (validated in the paper); the hardware speedup needs a custom kernel.\n      We provide both a reference fixed-butterfly FHT and the fast Re-Im path.\n\n  (B) Diagonal-capable LEARNED operator (preserves paper\'s elliptic win)\n      The learned spectral weights stay dense per-mode (exactly as HNO),\n      so the diagonal Green\'s-function multiplier remains representable.\n\n  (C) Optional structured CORRECTION as a regularizer (the new contribution)\n      A 2D butterfly-structured, cross-mode coupling term, GATED by a scalar\n      initialized to ZERO. At init, RHNO == HNO exactly. The correction can\n      only add structured coupling if it reduces loss. This is the honest\n      "regularization" story: structure is added on top of, not in place of,\n      the diagonal operator.\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport math\nfrom typing import Tuple, Optional\n\n\n# ---------------------------------------------------------------------------\n# 1. DHT (Re-Im FFT path) — fast, validated in paper\n# ---------------------------------------------------------------------------\n\ndef dht2d(x: torch.Tensor) -> torch.Tensor:\n    """2D DHT via FFT: H{f} = Re{F{f}} - Im{F{f}}."""\n    X = torch.fft.fft2(x)\n    return X.real - X.imag\n\n\ndef idht2d(H: torch.Tensor) -> torch.Tensor:\n    """Inverse 2D DHT (self-inverse up to 1/N)."""\n    N = H.shape[-1] * H.shape[-2]\n    return dht2d(H) / N\n\n\n# ---------------------------------------------------------------------------\n# 2. Dibit-reversal permutation (Jones Ch. 4) — valid permutation for any n\n# ---------------------------------------------------------------------------\n\ndef dibit_reverse_indices(n: int) -> torch.Tensor:\n    """\n    Valid permutation for any n.\n    Dibit-reversal for power-of-4, bit-reversal for power-of-2, identity else.\n    (The reordering only has structural meaning for power-of-2/4 sizes.)\n    """\n    if n == 1:\n        return torch.tensor([0], dtype=torch.long)\n    log4 = math.log(n, 4)\n    is_pow4 = abs(log4 - round(log4)) < 1e-9\n    log2 = math.log2(n)\n    is_pow2 = abs(log2 - round(log2)) < 1e-9\n\n    if is_pow4:\n        num_dibits = int(round(log4))\n        result = []\n        for i in range(n):\n            rev = 0; val = i\n            for _ in range(num_dibits):\n                rev = (rev << 2) | (val & 0x3); val >>= 2\n            result.append(rev)\n        return torch.tensor(result, dtype=torch.long)\n    elif is_pow2:\n        num_bits = int(round(log2))\n        result = []\n        for i in range(n):\n            rev = int(bin(i)[2:].zfill(num_bits)[::-1], 2)\n            result.append(rev)\n        return torch.tensor(result, dtype=torch.long)\n    else:\n        return torch.arange(n, dtype=torch.long)\n\n\n# ---------------------------------------------------------------------------\n# 3. Fixed-twiddle FHT (reference for the RFHT "fast transform" path)\n#\n# This computes the DHT using the radix-2 butterfly recurrence with FIXED\n# (non-learned) cas twiddle factors. It is mathematically identical to dht2d\n# but exposes the butterfly compute graph that maps to Jones\' FPGA architecture.\n# Used as a reference / for the hardware-deployment story, NOT as a learned op.\n# ---------------------------------------------------------------------------\n\ndef fht1d_fixed(x: torch.Tensor) -> torch.Tensor:\n    """\n    1D FHT along the last dimension using fixed cas twiddles.\n    Matches torch DHT (Re-Im) up to floating point. O(N log N) structure,\n    though implemented here with full matrices for clarity (reference only).\n    """\n    N = x.shape[-1]\n    n = torch.arange(N, device=x.device, dtype=x.dtype)\n    k = n.view(-1, 1)\n    cas = torch.cos(2 * math.pi * k * n / N) + torch.sin(2 * math.pi * k * n / N)\n    return torch.einsum(\'...n,kn->...k\', x, cas)\n\n\n# ---------------------------------------------------------------------------\n# 4. Optional 2D Butterfly Correction (the regularizer) — ZERO-INITIALIZED\n#\n# Adds structured cross-mode coupling on top of the diagonal operator.\n# Separable: a butterfly along height-modes and one along width-modes.\n# Gated by a learnable scalar `gamma` initialized to 0, so at init this is\n# a no-op and RHNO == HNO exactly.\n# ---------------------------------------------------------------------------\n\nclass ButterflyCorrection2d(nn.Module):\n    """\n    Zero-initialized structured correction. Couples modes within a quadrant\n    via learned 2x2 butterfly blocks along each spatial-frequency axis.\n    """\n    def __init__(self, channels: int, modes: int, num_stages: Optional[int] = None):\n        super().__init__()\n        self.channels = channels\n        self.modes = modes\n        if num_stages is None:\n            self.num_stages = max(1, int(math.log2(modes))) if modes > 1 else 1\n        else:\n            self.num_stages = num_stages\n\n        half = modes // 2\n        # Butterfly 2x2 blocks for height axis and width axis, per stage.\n        # Initialize near the identity butterfly [[1,1],[1,-1]]/sqrt(2) so the\n        # correction produces a MEANINGFUL signal h. The no-op property at init\n        # comes solely from gamma=0 (ReZero/LayerScale-style), which keeps the\n        # gradient dL/dgamma = <upstream, h> at a usable magnitude rather than\n        # vanishing (the bug from a tiny-weight init).\n        base = torch.tensor([[1.0, 1.0], [1.0, -1.0]]) / math.sqrt(2)\n        init_h = base.view(1, 1, 1, 2, 2).repeat(self.num_stages, half, channels, 1, 1)\n        init_w = init_h.clone()\n        init_h = init_h + 0.05 * torch.randn_like(init_h)\n        init_w = init_w + 0.05 * torch.randn_like(init_w)\n        self.bfly_h = nn.Parameter(init_h)\n        self.bfly_w = nn.Parameter(init_w)\n        # Zero-initialized gate — at init, correction contributes nothing,\n        # but its gradient is non-vanishing because h is meaningful.\n        self.gamma = nn.Parameter(torch.zeros(1))\n\n        self.register_buffer(\'perm\', dibit_reverse_indices(modes))\n\n    def _butterfly_axis(self, x: torch.Tensor, weights: torch.Tensor,\n                        axis: int) -> torch.Tensor:\n        """\n        Apply paired butterfly mixing along `axis` (either -2 or -1).\n        x: (B, C, m, m). weights: (num_stages, half, C, 2, 2).\n        """\n        h = x\n        m = self.modes\n        half = m // 2\n        for s in range(self.num_stages):\n            if axis == -1:\n                lo = h[..., :half]            # (B, C, m, half)\n                hi = h[..., half:]\n                pairs = torch.stack([lo, hi], dim=-1)        # (B,C,m,half,2)\n                # contract channel + pair-component with per-position 2x2\n                # weights[s]: (half, C, 2, 2)\n                out = torch.einsum(\'bcmhp,hcpq->bcmhq\', pairs, weights[s])\n                h = torch.cat([out[..., 0], out[..., 1]], dim=-1)\n                h = h[..., self.perm]\n            else:  # axis == -2\n                lo = h[:, :, :half, :]        # (B, C, half, m)\n                hi = h[:, :, half:, :]\n                pairs = torch.stack([lo, hi], dim=-1)        # (B,C,half,m,2)\n                out = torch.einsum(\'bchmp,hcpq->bchmq\', pairs, weights[s])\n                h = torch.cat([out[..., 0], out[..., 1]], dim=2)\n                h = h[:, :, self.perm, :]\n        return h\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """x: (B, C, modes, modes). Returns same shape, gated by gamma."""\n        h = self._butterfly_axis(x, self.bfly_w, axis=-1)\n        h = self._butterfly_axis(h, self.bfly_h, axis=-2)\n        return self.gamma * h\n\n\n# ---------------------------------------------------------------------------\n# 5. Corrected RFHT Spectral Convolution\n#\n# Main path: diagonal-capable dense per-mode weights (IDENTICAL to HNO),\n#            preserving the elliptic Green\'s-function advantage.\n# Optional:  zero-init butterfly correction adding structured coupling.\n# ---------------------------------------------------------------------------\n\nclass RFHTSpectralConv2d(nn.Module):\n    """\n    Corrected Hartley spectral convolution.\n\n    Parameters\n    ----------\n    in_channels, out_channels : int\n    modes : int\n        Modes retained per quadrant edge.\n    use_butterfly_correction : bool\n        If False, this is EXACTLY the HNO dense spectral conv.\n        If True, adds a zero-initialized structured correction (RHNO).\n    num_butterfly_stages : int, optional\n        Depth of the correction\'s butterfly (default log2(modes)).\n    """\n    def __init__(self, in_channels: int, out_channels: int, modes: int,\n                 use_butterfly_correction: bool = True,\n                 num_butterfly_stages: Optional[int] = None):\n        super().__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n        self.modes = modes\n        self.use_correction = use_butterfly_correction\n\n        # Diagonal-capable dense per-mode weights (4 quadrants x even/odd).\n        # einsum keeps modes (kh,kw) independent -> can represent any diagonal.\n        scale = 1.0 / (in_channels * out_channels)\n        self.W_even = nn.ParameterList([\n            nn.Parameter(scale * torch.randn(out_channels, in_channels, modes, modes))\n            for _ in range(4)\n        ])\n        self.W_odd = nn.ParameterList([\n            nn.Parameter(scale * torch.randn(out_channels, in_channels, modes, modes))\n            for _ in range(4)\n        ])\n\n        # Optional structured correction (operates in out_channel space)\n        if use_butterfly_correction:\n            self.correction = nn.ModuleList([\n                ButterflyCorrection2d(out_channels, modes, num_butterfly_stages)\n                for _ in range(4)\n            ])\n        else:\n            self.correction = None\n\n    def _diag_mul(self, W, x):\n        # (out,in,kh,kw),(B,in,kh,kw) -> (B,out,kh,kw): per-mode channel mix\n        return torch.einsum(\'oikj,bikj->bokj\', W, x)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        B, C, H, W = x.shape\n        m = self.modes\n\n        x_ht = dht2d(x)\n        x_flip = torch.roll(torch.flip(x_ht, dims=[-1, -2]),\n                            shifts=(1, 1), dims=[-1, -2])\n        x_even = (x_ht + x_flip) / 2.0\n        x_odd  = (x_ht - x_flip) / 2.0\n\n        slots = [\n            (slice(None, m),  slice(None, m)),\n            (slice(None, m),  slice(-m, None)),\n            (slice(-m, None), slice(None, m)),\n            (slice(-m, None), slice(-m, None)),\n        ]\n        quads_e = [x_even[:, :, sh, sw] for sh, sw in slots]\n        quads_o = [x_odd[:, :, sh, sw]  for sh, sw in slots]\n\n        out_ht = torch.zeros(B, self.out_channels, H, W,\n                             device=x.device, dtype=x.dtype)\n\n        for q, (sh, sw) in enumerate(slots):\n            # Diagonal-capable main operator (preserves elliptic advantage)\n            out = (self._diag_mul(self.W_even[q], quads_e[q]) +\n                   self._diag_mul(self.W_odd[q],  quads_o[q]))\n            # Optional zero-init structured correction\n            if self.correction is not None:\n                out = out + self.correction[q](out)\n            out_ht[:, :, sh, sw] = out\n\n        return idht2d(out_ht)\n'

RHNO_SRC = '"""\nRegularized Hartley Neural Operator (RHNO) — CORRECTED (v2)\n===========================================================\nBoth HNO and RHNO now share the SAME diagonal-capable spectral conv.\nThe only difference: RHNO enables a zero-initialized structured correction.\n\n  make_hno()  -> RFHTSpectralConv2d(use_butterfly_correction=False)\n                 == exact HNO from the paper (dense per-mode weights)\n  make_rhno() -> RFHTSpectralConv2d(use_butterfly_correction=True)\n                 == HNO + zero-init butterfly correction (regularizer)\n\nAt initialization, RHNO and HNO produce IDENTICAL outputs (gamma=0).\nThis guarantees RHNO can never start worse than HNO, and only adds\nstructured cross-mode coupling where it reduces loss.\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Optional\n\nfrom rfht import RFHTSpectralConv2d, dht2d, idht2d\n\n\n# Kept for backward-compatibility / direct comparison; identical to\n# RFHTSpectralConv2d(use_butterfly_correction=False).\nclass HNOSpectralConv2d(nn.Module):\n    """Original dense HNO spectral conv (paper Eq. 9)."""\n    def __init__(self, in_channels: int, out_channels: int, modes: int):\n        super().__init__()\n        self.conv = RFHTSpectralConv2d(\n            in_channels, out_channels, modes,\n            use_butterfly_correction=False\n        )\n\n    def forward(self, x):\n        return self.conv(x)\n\n\nclass SpectralBlock(nn.Module):\n    """One spectral conv block with residual bypass (paper Figure 1)."""\n    def __init__(self, channels: int, modes: int,\n                 use_rfht: bool = True,\n                 num_butterfly_stages: Optional[int] = None):\n        super().__init__()\n        # use_rfht == True  -> enable the structured correction (RHNO)\n        # use_rfht == False -> diagonal-only, == HNO\n        self.spectral_conv = RFHTSpectralConv2d(\n            channels, channels, modes,\n            use_butterfly_correction=use_rfht,\n            num_butterfly_stages=num_butterfly_stages\n        )\n        self.bypass = nn.Conv2d(channels, channels, kernel_size=1)\n        self.norm = nn.InstanceNorm2d(channels, affine=True)\n        self.act = nn.GELU()\n\n    def forward(self, x):\n        return self.act(self.norm(self.spectral_conv(x) + self.bypass(x)))\n\n\nclass HartleyNeuralOperator(nn.Module):\n    """\n    Hartley Neural Operator — base for HNO and RHNO.\n\n    use_rfht=False -> HNO  (diagonal-capable dense weights only)\n    use_rfht=True  -> RHNO (HNO + zero-init structured correction)\n    """\n    def __init__(\n        self,\n        in_channels: int = 4,\n        out_channels: int = 1,\n        width: int = 32,\n        modes: int = 16,\n        num_blocks: int = 3,\n        use_rfht: bool = True,\n        num_butterfly_stages: Optional[int] = None,\n    ):\n        super().__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n        self.width = width\n        self.modes = modes\n        self.num_blocks = num_blocks\n        self.use_rfht = use_rfht\n\n        self.input_proj = nn.Sequential(\n            nn.Linear(in_channels, width),\n            nn.GELU(),\n            nn.Linear(width, width),\n        )\n        self.blocks = nn.ModuleList([\n            SpectralBlock(width, modes, use_rfht=use_rfht,\n                          num_butterfly_stages=num_butterfly_stages)\n            for _ in range(num_blocks)\n        ])\n        self.output_proj = nn.Sequential(\n            nn.Linear(width, 128),\n            nn.GELU(),\n            nn.Linear(128, out_channels),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        h = self.input_proj(x)\n        h = h.permute(0, 3, 1, 2)\n        for block in self.blocks:\n            h = block(h)\n        h = h.permute(0, 2, 3, 1)\n        return self.output_proj(h)\n\n    def parameter_count(self) -> dict:\n        total = sum(p.numel() for p in self.parameters())\n        spectral = sum(\n            p.numel()\n            for block in self.blocks\n            for _, p in block.spectral_conv.named_parameters()\n        )\n        # Correction params specifically (the added regularizer cost)\n        corr = 0\n        for block in self.blocks:\n            if block.spectral_conv.correction is not None:\n                for c in block.spectral_conv.correction:\n                    corr += sum(p.numel() for p in c.parameters())\n        return {\n            \'total\': total,\n            \'spectral\': spectral,\n            \'correction\': corr,\n            \'non_spectral\': total - spectral,\n            \'spectral_fraction\': spectral / total,\n        }\n\n\ndef make_hno(in_channels=4, out_channels=1, width=32, modes=16,\n             num_blocks=3) -> HartleyNeuralOperator:\n    """Exact HNO from paper: diagonal-capable dense weights, no correction."""\n    return HartleyNeuralOperator(\n        in_channels=in_channels, out_channels=out_channels,\n        width=width, modes=modes, num_blocks=num_blocks,\n        use_rfht=False\n    )\n\n\ndef make_rhno(in_channels=4, out_channels=1, width=32, modes=16,\n              num_blocks=3, num_butterfly_stages=None) -> HartleyNeuralOperator:\n    """RHNO: HNO + zero-init structured (RFHT-inspired) correction."""\n    return HartleyNeuralOperator(\n        in_channels=in_channels, out_channels=out_channels,\n        width=width, modes=modes, num_blocks=num_blocks,\n        use_rfht=True, num_butterfly_stages=num_butterfly_stages\n    )\n'

with open(os.path.join(REPO_DIR, 'rfht.py'), 'w') as f:
    f.write(RFHT_SRC)
with open(os.path.join(REPO_DIR, 'rhno.py'), 'w') as f:
    f.write(RHNO_SRC)
print('Wrote corrected rfht.py and rhno.py')

# Fix gradient_error in training.py (torch.norm 3-dim bug)
tr_path = os.path.join(REPO_DIR, 'training.py')
with open(tr_path) as f:
    _s = f.read()
_old = '''    return (torch.norm(gp - gt, dim=(-1, -2, -3)) /
            torch.norm(gt, dim=(-1, -2, -3))).mean()'''
_new = ('    B = gp.shape[0]\n'
        '    gp = gp.reshape(B, -1); gt = gt.reshape(B, -1)\n'
        '    return (torch.linalg.norm(gp-gt,dim=1)/(torch.linalg.norm(gt,dim=1)+1e-8)).mean()')
if _old in _s:
    _s = _s.replace(_old, _new)
    with open(tr_path, 'w') as f:
        f.write(_s)
    print('Patched training.py gradient_error')
else:
    print('training.py gradient_error already OK')


## 3. Imports

In [ ]:
import torch
import numpy as np
import pickle
import math
import time
from torch.utils.data import DataLoader, random_split
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output

# Repo modules (flat structure)
from rfht import (
    dht2d, idht2d, dibit_reverse_indices,
    ButterflyStage, RFHTSpectralConv2d
)
from rhno import (
    HNOSpectralConv2d, HartleyNeuralOperator,
    make_hno, make_rhno
)
from training import (
    relative_l2, gradient_error,
    make_grf, make_eigenfunction_ic, make_gaussian_bump_ic,
    solve_heat_equation, solve_poisson,
    HeatDataset, PoissonDataset,
    train_model, eval_epoch
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")


## 4. Parameter Count Analysis

HNO uses diagonal-capable dense per-mode weights, which preserve the elliptic Green's-function advantage from the paper.

RHNO = HNO + a small zero-initialized structured correction. At init RHNO is identical to HNO; the correction engages only where cross-mode coupling reduces loss.


In [ ]:
print("=" * 72)
print("Parameter Count: HNO (diagonal dense) vs RHNO (HNO + zero-init correction)")
print("=" * 72)
print()
print("Corrected architecture:")
print("  HNO  = diagonal-capable dense per-mode weights (preserves elliptic win)")
print("  RHNO = HNO weights + small zero-initialized structured correction")
print("  At init, RHNO == HNO exactly (gamma=0); correction engages only if it helps.")
print()

configs = [
    (8,  32, 3, "small"),
    (16, 32, 3, "default"),
    (16, 48, 3, "wide"),
    (16, 32, 4, "elliptic-4blk"),
    (32, 32, 3, "high-mode"),
]

print(f"{'Config':<20} {'HNO Total':>11} {'RHNO Total':>11} "
      f"{'Correction':>11} {'Overhead':>9}")
print("-" * 65)

for modes, width, blocks, label in configs:
    hno = make_hno(width=width, modes=modes, num_blocks=blocks)
    rhno = make_rhno(width=width, modes=modes, num_blocks=blocks)
    hc = hno.parameter_count()
    rc = rhno.parameter_count()
    overhead = rc['total'] / hc['total']
    cfg = f"{label}(m={modes},w={width})"
    print(f"{cfg:<20} {hc['total']:>11,} {rc['total']:>11,} "
          f"{rc['correction']:>11,} {overhead:>8.3f}x")

print()
print("The correction adds only a few % parameters. The research question is")
print("whether that small structured term improves accuracy on PDEs where")
print("cross-mode coupling helps (broadband / nonlinear), while never hurting")
print("the elliptic case (guaranteed by the zero-init no-op property).")
print()

# Verify the no-op property numerically
import torch
torch.manual_seed(0)
m = make_rhno(in_channels=3, width=16, modes=16, num_blocks=3)
conv = m.blocks[0].spectral_conv
inp = torch.randn(2, 16, 32, 32)
conv.eval()
with torch.no_grad():
    out_on = conv(inp)
    saved = conv.correction; conv.correction = None
    out_off = conv(inp); conv.correction = saved
print(f"No-op check: max|RHNO_init - HNO| = {(out_on-out_off).abs().max().item():.2e} (== 0 confirms RHNO starts identical to HNO)")


## 5. Experiment Configuration

Adjust these to trade speed vs fidelity.  
**Quick run:** `RESOLUTION=64, N_SAMPLES=100, N_EPOCHS=50`  
**Paper setting:** `RESOLUTION=128, N_SAMPLES=200, N_EPOCHS=200`


In [ ]:
# ── Experiment config ──────────────────────────────────────────────────
RESOLUTION   = 64      # 128 for paper; 64 for quick
N_SAMPLES    = 100     # 200 for paper
N_EPOCHS     = 50      # 200 for paper
BATCH_SIZE   = 8
MODES        = 16      # power of 2 -> dibit-reversal active
WIDTH        = 32      # hidden channel width
BUTTERFLY_STAGES = None  # None = log2(modes); set int to ablate depth

# PDEs and IC types to run
EXPERIMENTS = [
    # (pde,       ic_type,        num_blocks, in_channels)
    ('poisson',   'grf',          4,          3),
    ('poisson',   'gaussian_bump',4,          3),
    ('heat',      'grf',          3,          4),
]

# HNO-optimized hyperparams (Table 3 of paper)
HNO_LR          = 3.8e-3
HNO_WD          = 1e-6
HNO_CLIP        = 5.0
HNO_SCHEDULER   = 'step'

print("Configuration:")
print(f"  Resolution:   {RESOLUTION}×{RESOLUTION}")
print(f"  Samples:      {N_SAMPLES}")
print(f"  Epochs:       {N_EPOCHS}")
print(f"  Modes:        {MODES}")
print(f"  Width:        {WIDTH}")
print(f"  Butterfly:    {BUTTERFLY_STAGES or f'log2({MODES})={max(1,int(math.log2(MODES)))} stages'}")
print(f"  Experiments:  {len(EXPERIMENTS)}")
print(f"  Device:       {DEVICE}")


## 6. Run HNO vs RHNO Experiments

Each experiment:
1. Generates PDE dataset (IC → solution pairs)
2. Trains HNO (dense, paper baseline) with paper-optimized hyperparams
3. Trains RHNO (structured-correction) with same hyperparams
4. Saves model checkpoints and training history to Drive


In [ ]:
all_results = {}

for pde, ic_type, num_blocks, in_channels in EXPERIMENTS:
    exp_key = f"{pde}_{ic_type}"
    print(f"\n{'='*65}")
    print(f"EXPERIMENT: {pde.upper()} / {ic_type}")
    print(f"{'='*65}")

    # ── Build dataset ──────────────────────────────────────────────────
    print("\nGenerating dataset...")
    t0 = time.time()

    if pde == 'poisson':
        dataset = PoissonDataset(
            n_samples=N_SAMPLES, resolution=RESOLUTION,
            ic_type=ic_type, device=DEVICE
        )
    elif pde == 'heat':
        dataset = HeatDataset(
            n_samples=N_SAMPLES, resolution=RESOLUTION,
            ic_type=ic_type, device=DEVICE
        )

    n_train = int(0.8 * len(dataset))
    n_test  = len(dataset) - n_train
    train_ds, test_ds = random_split(
        dataset, [n_train, n_test],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                               shuffle=False)

    print(f"  Train: {n_train}, Test: {n_test}  ({time.time()-t0:.1f}s)")

    exp_results = {}

    for model_name, use_rfht in [('HNO', False), ('RHNO', True)]:
        print(f"\n── Training {model_name} ──────────────────────────────────")
        t0 = time.time()

        model = HartleyNeuralOperator(
            in_channels=in_channels,
            out_channels=1,
            width=WIDTH,
            modes=MODES,
            num_blocks=num_blocks,
            use_rfht=use_rfht,
            num_butterfly_stages=BUTTERFLY_STAGES
        )
        counts = model.parameter_count()
        print(f"  Params: {counts['total']:,}  (spectral: {counts['spectral']:,})")

        history = train_model(
            model, train_loader, test_loader,
            n_epochs=N_EPOCHS,
            lr=HNO_LR, weight_decay=HNO_WD,
            clip_grad=HNO_CLIP,
            scheduler_type=HNO_SCHEDULER,
            device=DEVICE,
            verbose=True
        )
        elapsed = time.time() - t0
        print(f"  Training time: {elapsed:.0f}s")

        # Save checkpoint to Drive
        ckpt_path = os.path.join(SAVE_DIR, f"{exp_key}_{model_name}.pt")
        torch.save({
            'model_state_dict': model.cpu().state_dict(),
            'config': {
                'in_channels': in_channels,
                'out_channels': 1,
                'width': WIDTH,
                'modes': MODES,
                'num_blocks': num_blocks,
                'use_rfht': use_rfht,
                'num_butterfly_stages': BUTTERFLY_STAGES,
            },
            'history': history,
            'param_counts': counts,
            'pde': pde,
            'ic_type': ic_type,
            'resolution': RESOLUTION,
            'n_samples': N_SAMPLES,
            'n_epochs': N_EPOCHS,
            'training_time_s': elapsed,
        }, ckpt_path)
        print(f"  ✓ Checkpoint saved: {ckpt_path}")

        exp_results[model_name] = {
            'history': history,
            'counts': counts,
            'elapsed': elapsed,
        }

    all_results[exp_key] = exp_results

    # Quick summary
    hno_best  = exp_results['HNO']['history']['best_test_rel_l2']
    rhno_best = exp_results['RHNO']['history']['best_test_rel_l2']
    ratio     = rhno_best / hno_best
    print(f"\n  SUMMARY [{exp_key}]")
    print(f"    HNO  best Rel-L2: {hno_best:.6f}  "
          f"({exp_results['HNO']['counts']['total']:,} params)")
    print(f"    RHNO best Rel-L2: {rhno_best:.6f}  "
          f"({exp_results['RHNO']['counts']['total']:,} params)")
    print(f"    RHNO/HNO error ratio:  {ratio:.3f}x")
    print(f"    RHNO/HNO param ratio:  "
          f"{exp_results['RHNO']['counts']['total']/exp_results['HNO']['counts']['total']:.3f}x")

# Save all results summary as pkl
summary_path = os.path.join(SAVE_DIR, 'all_results_summary.pkl')
with open(summary_path, 'wb') as f:
    pickle.dump(all_results, f)
print(f"\n✓ Full results summary saved: {summary_path}")


## 7. Results Table

In [ ]:
print("\n" + "="*75)
print("RESULTS SUMMARY — HNO vs RHNO")
print("="*75)
print(f"{'Experiment':<22} {'Model':<6} {'Best Rel-L2':>12} {'Params':>10} "
      f"{'Spec Params':>12} {'RHNO/HNO':>10}")
print("-"*75)

for exp_key, exp_res in all_results.items():
    pde, ic = exp_key.split('_', 1)
    hno_best  = exp_res['HNO']['history']['best_test_rel_l2']
    rhno_best = exp_res['RHNO']['history']['best_test_rel_l2']

    for model_name in ['HNO', 'RHNO']:
        res = exp_res[model_name]
        best = res['history']['best_test_rel_l2']
        ratio_str = f"{rhno_best/hno_best:.3f}x" if model_name == 'RHNO' else "—"
        label = f"{pde}/{ic}" if model_name == 'HNO' else ""
        print(f"{label:<22} {model_name:<6} {best:>12.6f} "
              f"{res['counts']['total']:>10,} "
              f"{res['counts']['spectral']:>12,} "
              f"{ratio_str:>10}")

print()
print("RHNO/HNO < 1.0 = RHNO outperforms with fewer parameters")


## 8. Training Curves

In [ ]:
n_exp = len(all_results)
fig, axes = plt.subplots(n_exp, 2, figsize=(14, 4 * n_exp))
if n_exp == 1:
    axes = [axes]

colors = {'HNO': '#E05C5C', 'RHNO': '#4A90D9'}

for row, (exp_key, exp_res) in enumerate(all_results.items()):
    pde, ic = exp_key.split('_', 1)
    ax_loss, ax_test = axes[row]

    for model_name, res in exp_res.items():
        h = res['history']
        epochs = range(10, N_EPOCHS + 1, 10)
        epochs = list(epochs)[:len(h['train_loss'])]

        ax_loss.semilogy(epochs, h['train_loss'],
                          color=colors[model_name], alpha=0.4,
                          linestyle='--', linewidth=1)
        ax_test.semilogy(epochs, h['test_rel_l2'],
                          color=colors[model_name],
                          label=f"{model_name} ({res['counts']['total']:,} params)",
                          linewidth=2)

    ax_loss.set_title(f"{pde.upper()} / {ic} — Train MSE (log)")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Train Loss")
    ax_loss.grid(True, alpha=0.3)

    ax_test.set_title(f"{pde.upper()} / {ic} — Test Rel-L² (log)")
    ax_test.set_xlabel("Epoch")
    ax_test.set_ylabel("Rel-L² Error")
    ax_test.legend(fontsize=9)
    ax_test.grid(True, alpha=0.3)

plt.suptitle("HNO (dense) vs RHNO (butterfly-factorized)", fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()

plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved: {plot_path}")


## 9. Parameter Efficiency: Error vs Params

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

markers = {'HNO': 'o', 'RHNO': 's'}
sizes   = {'HNO': 100, 'RHNO': 100}

for exp_key, exp_res in all_results.items():
    pde, ic = exp_key.split('_', 1)
    label_base = f"{pde}/{ic}"

    for model_name, res in exp_res.items():
        best = res['history']['best_test_rel_l2']
        params = res['counts']['total']
        ax.scatter(params, best,
                   marker=markers[model_name],
                   s=sizes[model_name],
                   color=colors[model_name],
                   label=f"{model_name} – {label_base}",
                   zorder=3)
        ax.annotate(f"{model_name}\n{label_base}",
                    (params, best),
                    textcoords="offset points", xytext=(5, 5),
                    fontsize=7, alpha=0.8)

# Connect HNO→RHNO pairs
for exp_key, exp_res in all_results.items():
    hno_p  = exp_res['HNO']['counts']['total']
    hno_e  = exp_res['HNO']['history']['best_test_rel_l2']
    rhno_p = exp_res['RHNO']['counts']['total']
    rhno_e = exp_res['RHNO']['history']['best_test_rel_l2']
    ax.annotate('', xy=(rhno_p, rhno_e), xytext=(hno_p, hno_e),
                arrowprops=dict(arrowstyle='->', color='gray',
                                lw=1.2, alpha=0.5))

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel("Total Parameters", fontsize=11)
ax.set_ylabel("Best Test Rel-L² Error", fontsize=11)
ax.set_title("Parameter Efficiency: HNO vs RHNO\n"
             "(lower-left = better; arrows show HNO→RHNO reduction)",
             fontsize=11)
ax.grid(True, which='both', alpha=0.2)

# De-duplicate legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), fontsize=8,
          loc='upper right')

plt.tight_layout()
eff_path = os.path.join(SAVE_DIR, 'param_efficiency.png')
plt.savefig(eff_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved: {eff_path}")


## 10. Butterfly Depth Ablation  *(optional — runs additional training)*

Ablates the number of butterfly stages on the first experiment.  
Jones (2022) uses log₄(M) = log₂(M)/2 stages for radix-4 efficiency.  
This tests whether shallower = more regularized = better or worse.


In [ ]:
RUN_ABLATION = False  # Set True to run (takes ~2-3× training time)

if RUN_ABLATION:
    pde, ic_type, num_blocks, in_channels = EXPERIMENTS[0]
    exp_key = f"{pde}_{ic_type}"

    dataset = PoissonDataset(n_samples=N_SAMPLES, resolution=RESOLUTION,
                              ic_type=ic_type, device=DEVICE)
    n_train = int(0.8 * len(dataset))
    train_ds, test_ds = random_split(
        dataset, [n_train, len(dataset)-n_train],
        generator=torch.Generator().manual_seed(42)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                               shuffle=True, drop_last=True)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    ablation_results = {}
    max_stages = max(1, int(math.log2(MODES)))

    for stages in range(1, max_stages + 1):
        print(f"\n── RHNO depth ablation: {stages} stage(s) ──────────────")
        model = make_rhno(in_channels=in_channels, width=WIDTH,
                           modes=MODES, num_blocks=num_blocks,
                           num_butterfly_stages=stages)
        counts = model.parameter_count()
        print(f"  Params: {counts['total']:,}")
        history = train_model(model, train_loader, test_loader,
                               n_epochs=N_EPOCHS, lr=HNO_LR,
                               weight_decay=HNO_WD, clip_grad=HNO_CLIP,
                               scheduler_type=HNO_SCHEDULER,
                               device=DEVICE, verbose=False)
        ablation_results[stages] = {
            'history': history, 'counts': counts
        }
        print(f"  Best Rel-L2: {history['best_test_rel_l2']:.6f}")

    ablation_path = os.path.join(SAVE_DIR, f'ablation_depth_{exp_key}.pkl')
    with open(ablation_path, 'wb') as f:
        pickle.dump(ablation_results, f)
    print(f"\n✓ Ablation results saved: {ablation_path}")

    # Plot
    stages_list = sorted(ablation_results.keys())
    errors  = [ablation_results[s]['history']['best_test_rel_l2'] for s in stages_list]
    params  = [ablation_results[s]['counts']['total'] for s in stages_list]
    hno_ref = all_results[exp_key]['HNO']['history']['best_test_rel_l2']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(stages_list, errors, 'o-', color='#4A90D9', linewidth=2, markersize=8)
    ax1.axhline(hno_ref, color='#E05C5C', linestyle='--', linewidth=1.5,
                label=f'HNO baseline ({hno_ref:.4f})')
    ax1.set_xlabel("Butterfly Stages")
    ax1.set_ylabel("Best Test Rel-L²")
    ax1.set_title(f"Depth Ablation: {pde.upper()}/{ic_type}")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(stages_list, params, 's-', color='#4A90D9', linewidth=2, markersize=8)
    ax2.axhline(all_results[exp_key]['HNO']['counts']['total'],
                color='#E05C5C', linestyle='--', linewidth=1.5, label='HNO params')
    ax2.set_xlabel("Butterfly Stages")
    ax2.set_ylabel("Total Parameters")
    ax2.set_title("Parameter Count vs Depth")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'ablation_depth.png'), dpi=150)
    plt.show()
else:
    print("Skipped (set RUN_ABLATION = True to run).")


## 11. Saved Files Summary

In [ ]:
import os

print(f"Files saved to {SAVE_DIR}:\n")
total_bytes = 0
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath)
    total_bytes += size
    ext   = fname.split('.')[-1].upper()
    print(f"  [{ext:4s}] {fname:<45}  {size/1024/1024:.2f} MB")

print(f"\nTotal: {total_bytes/1024/1024:.2f} MB")
print()
print("To reload a checkpoint later:")
print("  ckpt = torch.load('/content/drive/MyDrive/RHNO_experiments/poisson_grf_HNO.pt')")
print("  model = HartleyNeuralOperator(**ckpt['config'])")
print("  model.load_state_dict(ckpt['model_state_dict'])")
print()
print("To reload results summary:")
print("  with open('.../all_results_summary.pkl', 'rb') as f:")
print("      results = pickle.load(f)")
